# Session 10 · The model landscape
## Choosing a model: catalog, cost, speed, quality, and the selection matrix

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 6. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

**The task in one sentence.** The loan assistant does three different jobs, and each needs a different model. You fill in the evidence, then choose one model per job and defend it.

**How the decision is made (memorise this order):** 1) **hard constraints** eliminate models (data must stay in-house, latency limit, context too small); 2) among survivors, a **weighted score** of cost, speed and quality ranks them; 3) you **verify** the winner on your own data (S17 gives you the tool).

**Before you start:** in PowerShell, in the course folder: `git pull`, then `uv sync` (no new packages today). Open the notebook in VS Code, or run `uv run jupyter lab module01/s10_model_matrix.ipynb`. Run it top to bottom (Kernel → Restart & Run All). Only section 4 talks to a model. With no key and no Ollama, the notebook still runs top to bottom: section 4 prints `skipped (...)` and everything else works, because the catalog, the costs and the matrix are pure arithmetic.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A bank builds a loan assistant. The team reads a leaderboard, picks the model at the top, and uses it for everything. The assistant does three jobs: it answers **public FAQ** questions about loan products, it **reads customers' documents** (salary slips and bank statements), and it gives **live suggestions to call-centre agents** while a customer is on the phone. Within two months, three things go wrong.

1. **The bill.** The FAQ bot gets about 20,000 questions a day. On a flagship-tier model, the course catalog's indicative prices give about **Rs 5,54,400 a month** for this one job. A small hosted open-weight model, which answers simple product questions well enough, would cost about **Rs 7,560**. That is roughly 73 times less, for the same job.
2. **Compliance stops the document reader.** Salary slips contain names, account numbers and salaries. The flagship runs only on the vendor's servers, so every slip leaves the bank. The compliance team rules that this data must stay in-house. The project is paused. No price discount can fix this: the model was never allowed for this job.
3. **The agents stop using the suggestions.** The flagship is a reasoning model. It thinks before it answers, and in the catalog's starting guess it needs about **6 seconds**. By then the customer has asked the next question. The call-centre job needed an answer in about **3 seconds**.

Then someone says: "Let's run a free model on a laptop instead." It costs nothing per token. But one laptop answers one request at a time. It cannot serve 20,000 calls a day.

None of these is a "bad model" problem. **Each one is a selection problem: the wrong model for the job, chosen without a method.** Today you learn the method, and you use it on exactly these three jobs.

> The numbers above come from `models_catalog.json` and the three scenarios in `landscape_utils.py`. They are **indicative** (the catalog says so). You will verify some of them yourself today.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Name the **three ways to get a model**: vendor API, hosted open-weight, self-hosted | Sending customer data to someone else's machine without noticing | Restaurant, cloud kitchen, home kitchen |
| Compute the **monthly cost** of a job from token prices and volume | A bill 70 times larger than it needed to be | Price the task, not the token |
| Apply **hard constraints** before comparing anything | Choosing a cheap or clever model that is not allowed, too slow, or cannot handle the volume | Pass or out |
| Rank survivors with a **weighted score** whose weights the business sets | Engineers quietly deciding business priorities | Who chooses the weights matters |
| **Measure** speed and quality on your own prompt, instead of trusting a catalog or a leaderboard | Surprises in production | Catalog numbers are a prior; your measurements are evidence |

### Where this is used in industry
- **Banking and insurance:** a new AI use case usually goes through model-risk and vendor-risk review. Reviewers ask *why this model*, *where does the data go*, and *what does it cost at our volume*. A written selection matrix with reasons answers all three.
- **Healthcare:** patient records are often restricted to approved environments, so the choice is usually between a self-hosted open-weight model and a private endpoint that compliance has approved. Residency removes options first.
- **Contact centres in telecom and retail:** agent-assist and live chat have a latency budget. A model that is too slow is out, however good its answers are.
- **Cost control at scale:** teams estimate the monthly bill before building, and later route easy requests to a cheap tier and hard ones to an expensive tier (Session 54).
- **Platform teams:** keep an internal catalog of approved models with prices, context windows and allowed data classes, and re-check it often, because prices and models change every few months.

### Why interviewers care
These questions come up in AI-engineer and GenAI-developer interviews:
- *"How would you choose an LLM for a new use case?"*
- *"What is the difference between open-weight, open-source and closed models? When would you self-host?"*
- *"How do you estimate the monthly cost of an LLM feature?"*
- *"Can you trust a leaderboard to pick a model?"*
- *"Which latency numbers matter for a chat application, and how does a reasoning model change them?"*

By the end of today you can answer each one with a definition, an example from your own lab, and one honest limit. Part C has model answers.

<details><summary><b>▸ Read more: why "the best model" is the wrong question</b></summary>

"Best" has no meaning without a job. A public FAQ bot needs to be cheap at high volume; a document reader needs quality and must keep data in-house; an agent-assist tool needs speed. These goals pull in different directions, so one model rarely wins all three. Experienced teams therefore keep a short list of approved models and choose per job. They also expect to change their choice later: the rest of the system (prompts, retrieval, tools) stays the same when the model in the middle changes. That is why the course teaches a *method*, not a list. The list will be out of date in a few months; the method will not.
</details>

## A2. Where this session fits in the course

```
 S09  How LLMs work ───────────► 9a: your first call; 9b: tokens, the context window, cost per call
  │                              (today reuses chat_client() and ollama_client() from tokens_utils.py)
 S10  The model landscape ─────► TODAY: which model for which job; eliminate → score → verify
  │
 S11  Multi-provider calls ────► one ask() for Groq, Gemini and Ollama; temperature, max_tokens, stop
  │
 S12  Local and open models ───► Ollama in depth; quantization; when policy requires local inference
  │
 S13  Multimodal + structured ─► images and JSON output (the vision column of today's catalog)
  │
 S17  Prompt evaluation ───────► the "verify" step done properly: your own golden set
  │
 S22  Serving · S53 Observability · S54 LLMOps ──► streaming and TTFT, cost tracking, routing by tier
 S57  Governance · S58–S60 Cloud AI ─────────────► data residency and private endpoints at enterprise scale
```

**What you need from earlier sessions:**
- `uv sync` done and `uv run pytest tests/test_setup.py` green (Session 2).
- `.env` in the course folder with your Groq key in `API_KEY`, plus `BASE_URL` and `MODEL` as in `.env.example` (Session 2). Do not change `MODEL`; the instructor pins it.
- Tokens, the context window, and the cost of one call (Session 9). Today multiplies that cost by a whole month.
- Optional: Ollama with `ollama pull llama3.2:3b` done. Without it, half of section 4 is skipped and everything else works.

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B. The same list, in short form, is in `module01/terms_s10.md`.

---
**1. Vendor API (closed model)**
- **Definition:** a model that only its vendor runs. You send your text over the internet and pay per token. You cannot download the model.
- **Analogy:** a big-brand restaurant. Their chef, their kitchen, their recipe. You order and pay per plate; you never see the recipe.
- **Example:** the GPT, Claude and Gemini families. In today's catalog: the three `Frontier ... (closed, vendor API)` rows.
- **Often confused with:** "any model you call over the internet". Hosted open-weight models are also called over the internet, but their weights are public.
- **Comes back in:** S11, S37, S58–S60.

---
**2. Frontier model, family and tier**
- **Definition:** *frontier* means the most capable models at a given moment, mostly closed. A *family* is one vendor's line-up; a *tier* is one size in it: flagship, mid-tier, economy. Same brand, very different price and speed.
- **Analogy:** a car maker's range: hatchback, sedan, SUV. Nobody takes a truck to buy vegetables.
- **Example:** the catalog's flagship row costs 5.00 USD in / 25.00 USD out per million tokens; the economy row 0.50 / 2.50 (indicative).
- **Often confused with:** "the flagship is always better for my task". Start with the smallest tier and move up only when your own test says you must.
- **Comes back in:** S37, S54 (routing easy requests to cheap tiers).

---
**3. Open-weight (and open-source)**
- **Definition:** the trained numbers (the *weights*) are published and anyone can download them. *Open-source* would also release the code and the training data; most open-weight models release only the weights, under a licence that may restrict use.
- **Analogy:** a published recipe. Anyone can cook the dish. Read the licence: some recipes say "not for commercial kitchens".
- **Example:** gpt-oss, Llama, Qwen, Mistral, DeepSeek. Column `open` in the catalog.
- **Often confused with:** "free". The recipe is free; the cooking is not (a provider's per-token price, or your own hardware and people).
- **Comes back in:** S12, S57.

---
**4. Hosted open-weight vs self-hosted (on-premises)**
- **Definition:** *hosted open-weight*: a provider runs published weights for you and bills per token; your text goes to the provider. *Self-hosted*: you run the same weights on your own machine (a laptop with Ollama, or a GPU server in your data centre); your text never leaves.
- **Analogy:** a cloud kitchen cooking the published recipe for you, versus cooking it in your own home kitchen.
- **Example:** `gpt-oss-20b on Groq` (hosted) and `gpt-oss-20b self-hosted on a smaller GPU server`: the same weights, two different rows, two different costs, two different data rules.
- **Often confused with:** two different models. They are one model in two places. Where a model runs is a separate decision from which model.
- **Comes back in:** S11, S12, S57, Module 12.

---
**5. Parameters**
- **Definition:** the learned numbers inside a model. "3B, 20B, 120B" means 3, 20 or 120 billion of them.
- **Analogy:** knobs tuned during training. More knobs usually means more capable, slower, and more memory.
- **Example:** a rule of thumb from the deck: about 2 bytes per parameter at full precision, so a 20B model is roughly 40 GB; squeezed to 4-bit, roughly 10–12 GB (that squeeze is quantization, Session 12).
- **Comes back in:** S12 (which model fits which laptop), S54.

---
**6. Mixture-of-experts (MoE)**
- **Definition:** a design with many parameters in total, but only a fraction of them active for each token.
- **Analogy:** a big hospital with a hundred specialists; each patient sees only the two or three relevant ones. The hospital still needs room for all hundred (memory), but each patient is quick (speed).
- **Example:** gpt-oss and Llama 4 Scout (`17B-16E` in its name) are built this way, which is why they feel fast for their size.
- **Often confused with:** a small model. You still need memory for all the parameters.
- **Comes back in:** S12.

---
**7. Context window and max output tokens**
- **Definition:** the *context window* is how many tokens the model can see at once, input and output together. *Max output tokens* is a separate, smaller limit on one answer.
- **Analogy:** the size of the desk (Session 9), and a page limit for the reply.
- **Example:** column `ctx` in the catalog: 131072 for most rows, 1000000 for the economy-tier row. The scenarios need at least 8,000 or 16,000 (`min_ctx`).
- **Often confused with:** "a big window is free". You pay for every input token on every call. And Ollama's *default* window is far smaller than the model's maximum unless you raise `num_ctx`.
- **Comes back in:** S11 (`max_tokens`), S19, S30, S38.

---
**8. Price per million tokens (input and output)**
- **Definition:** how providers bill: one price for input tokens and a different, usually higher, price for output tokens, each quoted in USD per million tokens.
- **Analogy:** two prices on the menu: one for what you bring to the table, one for what the kitchen sends back.
- **Example:** gpt-oss-20b on Groq in the catalog: 0.075 USD in, 0.30 USD out per million.
- **Often confused with:** the cost of a finished task. A cheap model that needs three attempts can cost more than a pricier one that is right the first time. Caching repeated prompt beginnings and batch jobs can also lower the price.
- **Comes back in:** S38, S53 (cost tracking), S54.

---
**9. Reasoning tokens**
- **Definition:** hidden "thinking" tokens a reasoning model writes before the visible answer. You do not see them, but they are billed as output and they take time.
- **Analogy:** rough work on a spare sheet before writing the answer. You never see the rough sheet, but you pay for the paper.
- **Example:** column `reasoning` in the catalog. The course model, gpt-oss-20b, is a reasoning model.
- **Comes back in:** S11 (an empty reply when `max_tokens` is too low), S15, S34.

---
**10. Time to first token (TTFT)**
- **Definition:** the delay between sending a request and seeing the first visible token. For a reasoning model it includes the hidden thinking.
- **Analogy:** how long until the first dish reaches your table.
- **Example:** section 4 prints `first token ... s` for Groq and for your laptop.
- **Often confused with:** total time. A chat user feels TTFT; a call-centre agent waiting for a full suggestion feels the total.
- **Comes back in:** S22 (streaming), S53 (latency dashboards).

---
**11. Tokens per second (and total time)**
- **Definition:** output speed once the answer is coming. *Total time* is TTFT plus the time to produce the whole answer.
- **Analogy:** how fast the remaining courses arrive, and how long until the meal is over.
- **Example:** hosted models on fast hardware produce hundreds of tokens per second; a small laptop model produces tens.
- **Often confused with:** a property of the model alone. Speed depends on the model *and* the hardware *and* the load. "Fast" is a property of a model in a place.
- **Comes back in:** S12, S22, S54.

---
**12. Benchmark, leaderboard, contamination**
- **Definition:** a *benchmark* is a public test; a *leaderboard* ranks models on it. *Contamination* means a model has seen the test questions during training.
- **Analogy:** a competitive-exam rank. Good for deciding whom to interview, not whom to hire. Contamination is a paper leak.
- **Example:** a model that tops an English reasoning test may still answer Telugu banking questions badly. You only know by asking it.
- **Often confused with:** a verdict. Use leaderboards to shortlist; decide with your own test.
- **Comes back in:** S17 (your own golden set), S52.

---
**13. Hard constraint**
- **Definition:** a requirement that removes a model *before* any scoring: the data must stay in-house, the answer must come within 3 seconds, the context must fit, the volume must be served, the quality must reach a minimum.
- **Analogy:** eligibility for a job: no degree, no interview, however charming the candidate.
- **Example:** in the `documents` job, `needs_on_prem=True` removes every hosted and vendor model, with the reason `data must stay on-prem; this model's provider sees the prompt`.
- **Often confused with:** a heavy weight. A weight only moves a score up or down; a hard constraint removes the option completely.
- **Comes back in:** S37, S40, S61 (capstone design).

---
**14. Data residency**
- **Definition:** rules about where data may be processed or stored (which building, which country, which approved environment). Compliance and the law set them, not engineers.
- **Analogy:** "Where may this file go?"
- **Example:** customer salary slips: in-house or an approved private deployment. Public product FAQs: anywhere.
- **Comes back in:** S57, Module 12.

---
**15. Weighted score (and normalisation)**
- **Definition:** cost, speed and quality, each turned into a number from 0 (worst) to 1 (best) among the surviving models (*normalisation*), then combined with weights that add up to 1, and shown as 0–100.
- **Analogy:** an interview panel's scorecard. The job description decides how much each part counts.
- **Example:** `faq` weights are cost 0.6, latency 0.2, quality 0.2; `documents` weights are cost 0.2, latency 0.2, quality 0.6.
- **Often confused with:** an objective truth. Change the weights and the winner moves, so someone accountable must choose them.
- **Comes back in:** S37, S54.

---
**16. Catalog value vs measurement (prior vs evidence)**
- **Definition:** a *catalog value* is a starting guess written down by someone else (here: the instructor's quality and latency guesses, and tracker prices). A *measurement* is what you observe on your own prompt, machine and network.
- **Analogy:** the restaurant review versus the meal you ate.
- **Example:** in section 4 you time gpt-oss-20b and llama3.2:3b, score their answers, and your numbers replace the catalog's guesses for those two rows (`MEASURED`).
- **Comes back in:** S17, S52, S53.

## A4. How this session is built

### The big picture

```
┌── COURSE REPOSITORY  (you pull it; you do not edit it) ──────────────────────────────────────────────────┐
│                                                                                                          │
│ module01/models_catalog.json   THE DATA: 11 model rows, indicative values (as_of 2026-09)                │
│    id · name · access · open_weight · runs_on_prem · ctx_tokens · price_in · price_out (USD per 1M)      │
│    fixed_monthly_usd · reasoning · vision · quality (1-5 guess) · latency_s (guess) · notes              │
│                                                                                                          │
│ module01/landscape_utils.py    THE TOOLS (no network, except time_stream)                                │
│    load_catalog · SCENARIOS (faq, documents, agent_assist) · USD_TO_INR = 84 · DAYS = 30                 │
│    monthly_cost_usd / monthly_cost_inr        price × tokens × calls, plus any fixed monthly cost        │
│    eligible · rank                            hard constraints first, then the weighted score            │
│    PROMPT · emi · check_answer · RUBRIC        one EMI question with a checkable answer                  │
│    time_stream(client, model)                  one streamed answer, timed                                │
│                                                                                                          │
│ module01/tokens_utils.py       FROM SESSION 9 (reads .env on import: load_dotenv(override=True))         │
│    chat_client()  -> the provider in .env (Groq)      ollama_client() -> localhost:11434/v1              │
│                                                                                                          │
│ module01/s10_model_matrix.ipynb  ◄── YOU ARE HERE                                                        │
│ module01/reading_landscape.md, terms_s10.md   the pre-read and the terms table                           │
│ tests/test_s10.py              the checkpoint: 8 tests, all offline (no key, no Ollama)                  │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                         │ the notebook imports the tools; tokens_utils reads ▼ your .env
┌── YOUR MACHINE  (yours; never committed) ────────────────────────────────────────────────────────────────┐
│ .env           BASE_URL · API_KEY · MODEL (Groq)            used only in section 4                       │
│ Ollama         optional; listens on http://localhost:11434 ; model llama3.2:3b                           │
│ module01/s10_matrix_output.md   written by section 6; you upload it through the submission form          │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                         │ only section 4 sends requests ▼ (two streamed answers)
                 Groq API (api.groq.com)                          Ollama (your laptop)
```

Everything except section 4 is arithmetic on a JSON file: no key, no internet, the same result on every laptop. That is deliberate. The *method* should not depend on a network; only the *measurement* does.

### What happens to one scenario

```
 SCENARIOS["documents"]                  models_catalog.json                 OVERRIDES
 2,000 calls/day · 6,000 in / 500 out    11 rows of starting values         your filled price + MEASURED rows
 needs_on_prem=True · min_quality 3            │                                    │
        │                                      │                                    │
        └───────────────────► rank(scenario, overrides=OVERRIDES) ◄─────────────────┘
                                               │
   1. for each row: merge your override        m = {**row, **override}
   2. eligible(): the hard constraints         on-prem? latency? volume? context? quality? price known?
         │ fails one or more ──► EXCLUDED, with every reason  ("data must stay on-prem; ...")
         ▼ passes all
   3. monthly cost (USD) for each survivor     calls × tokens × price / 1,000,000 + fixed monthly cost
   4. normalise AMONG THE SURVIVORS            cost (log scale), latency, quality ──► each 0..1, 1 = best
   5. score = 100 × (w_cost × cost + w_latency × latency + w_quality × quality)
         ▼
   ranked list (best first) + excluded list  ──► printed in section 5, saved to a file in section 6
```

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `chat_client()` | tokens_utils | nothing (reads `BASE_URL`, `API_KEY` from `.env`) | a client for the provider in `.env` (Groq); raises an error if there is no key at all | 1 |
| `ollama_client()` | tokens_utils | nothing | a client for `localhost:11434/v1` (does not check that Ollama runs) | 1 |
| `EXPECTED_EMI`, `EXPECTED_RATIO` | landscape_utils | (constants) | the right answers to `PROMPT`: 17594.09 and 42.7 | 1, 4, 6 |
| `load_catalog()` | landscape_utils | nothing | the 11 rows of `models_catalog.json`, as a list of dictionaries | 2 |
| `SCENARIOS` | landscape_utils | (data) | the three jobs: volume, tokens per call, hard limits, weights | 3, 5, 6 |
| `monthly_cost_inr(model, scenario)` | landscape_utils | one row, one scenario | rupees per month (rounded), or `None` if a price is unknown | 3 |
| `PROMPT`, `RUBRIC` | landscape_utils | (text) | the EMI question; the 1–5 quality rubric | 4 |
| `time_stream(client, model)` | landscape_utils | a client and a model name | `{"ttft_s", "total_s", "completion_tokens", "tokens_per_s", "text"}` | 4 |
| `check_answer(text)` | landscape_utils | the model's answer | `{"emi_ok", "ratio_ok", "score_out_of_2"}` | 4 |
| `rank(scenario, overrides=...)` | landscape_utils | one scenario and your overrides | `(ranked, excluded)`: survivors with `score`, `cost_inr`; and `(name, reasons)` pairs | 5, 6 |
| `eligible(model, scenario, overrides)` | landscape_utils | one row, one scenario | `(True/False, list of reasons)`; called inside `rank` | 5 (inside `rank`) |

Three **under the hood** cells rebuild the cost formula, the answer checker and the scoring rule in front of you, and check they give exactly the same results as the real helpers, with no model call.

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup | no (creates two clients; sends nothing) | 2 |
| 2 | The catalog — ten rows, one table | no | 6 |
| 3 | Three jobs, three bills | no (arithmetic) | 6 |
| 4 | Measure — the same prompt on a hosted model and on your laptop | yes, 2 streamed answers (Groq and Ollama), up to 1,500 output tokens each | 10 |
| 5 | The selection matrix | no | 8 |
| 6 | Your decision — this is what you submit | no (writes one file) | 10 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Classify any model as vendor API, hosted open-weight or self-hosted, and say who sees the data in each case.
2. Read a model catalog row (access, open weights, context, prices, reasoning) and spot what is a fact, what is a guess, and what is missing.
3. Compute the monthly cost of a job from calls per day, tokens per call and the price card, and explain why a self-hosted row has a 0 per-token price but a monthly cost that is not zero.
4. Measure time to first token, total time and tokens per second on a hosted and a local model, and score the answers with a rubric.
5. Apply hard constraints, then a weighted score, and explain why a model was excluded or ranked first.
6. Choose one model per job, name a runner-up, and state what evidence would change your mind.

**What you submit:**

| What | Where | How it is checked |
|---|---|---|
| Your decision for each job, the two verified catalog rows, answers to Questions 1–5 | section 6 of this notebook | read by the instructor in the debrief |
| `module01/s10_matrix_output.md` | written by the last cell; upload it through the submission form (personal repositories start at Session 14) | the instructor reads your ranked lists and measurements |
| The checkpoint | terminal: `uv run pytest tests/test_s10.py` | 8 tests, all offline: `8 passed` |

**The checkpoint:** the S10 tests need no key and no Ollama, so everyone should see `8 passed`. They check the helpers (catalog, cost formula, constraints, scoring, EMI checker), not your answers: your answers are in section 6 and in the uploaded file.

**Budget:** very light. Section 4 sends one request to Groq and one to your laptop. `time_stream` allows up to 1,500 output tokens, because the course model is a reasoning model and needs room to think before it writes. If the whole class runs section 4 at the same moment, a free-tier `429` (rate limit) is possible: wait 30 seconds and re-run that cell only.

---
# Part B · Hands-on

## 1. Setup

### What and why
`landscape_utils.py` holds the catalog loader, the three scenarios, the cost formula, the constraint filter, the scorer, and the timing helper. `tokens_utils.py` (from Session 9) gives us the two clients: Groq from `.env`, and Ollama on your laptop.

Some of you have a Groq key and Ollama; some have only the key; a few have neither yet. The setup must cope with all of these: say clearly what it found, and let the notebook continue. Only section 4 needs a client; everything else is arithmetic.

### The idea: two clients, one comparison
```
   chat_client()  ──► Groq (from .env)           hosted open-weight: your text goes to the provider
   ollama_client() ─► localhost:11434 (laptop)   self-hosted: your text never leaves your machine
                         │
                         └── the SAME prompt goes to both in section 4, so the comparison is fair
```
**Analogy — the cloud kitchen and the home kitchen.** Same kind of dish ordered from two places: one fast and professional, one in your own kitchen where nobody else touches the ingredients. Today you taste both and time both.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the *front* of Python's search list (`[:0]` is a slice meaning "before the first item"), so the imports work whether Jupyter runs from the repo root or from `module01/`.
2. **`from landscape_utils import *`** imports every public name from the helper file at once: `load_catalog`, `SCENARIOS`, `rank`, `PROMPT`, `EXPECTED_EMI` and the rest. (`*` is convenient in a teaching notebook; in production code you list the names, so a reader knows where each one comes from.)
3. **Importing `tokens_utils` reads your `.env`**: it calls `load_dotenv(override=True)`, so `os.getenv("MODEL")` sees your settings.
4. `ollama = ollama_client()` creates the local client. This never fails, even with Ollama stopped, because creating a client sends nothing.
5. **`try: ... except Exception as e:`** tries to create the Groq client. With no key at all, the `openai` library raises an error; the `except` part catches it, sets `groq = None`, and prints the first 80 characters of the message (`str(e)[:80]`). Section 4 then skips Groq. This is why this cell needed **no change** for the no-key rule.
6. The last line prints the hosted model name and the two right answers to today's EMI question, computed in Python, not by a model.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client
ollama = ollama_client()
try:
    groq = chat_client()                      # Groq, from .env
except Exception as e:                        # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

### What you should see
With a key in `.env`, one line. The two numbers come from Python arithmetic, so they are the same on every laptop; only the model name depends on your `.env`:
```
hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %
```
With no `.env` (or no `API_KEY` in it), two lines. The first is the library's message, cut at 80 characters, for example (*an illustration*; the exact wording depends on the `openai` version):
```
no hosted client: Missing credentials. Please pass an `api_key`, ...
hosted model: None | expected EMI: 17594.09 | expected ratio: 42.7 %
```

| If you see... | It means... | Do this |
|---|---|---|
| `no hosted client: ...` but you have a key | `.env` is missing from the course folder, or the key line is empty | fix `.env` (not `.env.example`), save, **restart the kernel** |
| `hosted model: None` but a key works | `MODEL=` is missing in `.env` | copy the `MODEL` line from `.env.example`; restart |
| No error, but your key is still `paste_your_groq_key_here` | a placeholder is still a non-empty value, so the client is created | section 4 will print `skipped (AuthenticationError ...)`; paste your real key |
| `ModuleNotFoundError: landscape_utils` | repo out of date, or the notebook was opened from another folder | `git pull`; open the notebook from the course folder |

### Check yourself
*(New question.) The setup created an Ollama client without any error, even on a laptop where Ollama is not installed. Why? In which section will you first find out whether Ollama really works?*

> **Common mistake:** editing `.env` and re-running only this cell. `tokens_utils` read `.env` when it was first imported; Python does not import a module twice. Restart the kernel after any change to `.env`, then run from the top.

## 2. The catalog — ten rows, one table

### What and why
Read `models_catalog.json` as a table. **Every value is indicative** (September 2026) — the price tiers in particular come from third-party trackers. `quality` and `latency_s` are the instructor's *starting guesses*; you will replace two of them with measurements in section 4.

(The title says ten rows; today's file has eleven. A catalog grows and changes. That is the point of this section.)

Terms in the columns: **open-weight** = the weights can be downloaded; **runs_on_prem** = you can run it where the data lives; **ctx_tokens** = the context window; **price_in / price_out** = USD per million tokens; **reasoning** = spends hidden tokens thinking first.

### The idea: three ways to get a model, one table
Every row in the catalog is one of three kinds. The column `access` tells you which:

```
  access = vendor-api            access = hosted-api              access = local / self-hosted
  ───────────────────            ───────────────────              ────────────────────────────
  the RESTAURANT                 the CLOUD KITCHEN                the HOME KITCHEN
  vendor's recipe, vendor's      published recipe (open weights)  same published recipe,
  kitchen                        cooked by a provider (Groq)      cooked by you (Ollama, GPU server)
  your text → the vendor         your text → the provider         your text → nowhere
  you pay per token              you pay per token (usually less) you pay hardware + people
          ▲                               ▲                                ▲
          └──────── data crosses the building's wall ──────────┘    data stays inside
```
Three questions to ask about any row: **who sees my data, who sends the bill, and who is on call when it breaks?**

Two rows deserve a second look. `gpt-oss-20b on Groq` and `gpt-oss-20b self-hosted on a smaller GPU server` are the **same weights in two places**. The catalog gives them different prices, different latency guesses, and a different `runs_on_prem`. In section 5 that single column decides one whole job.

### What the next cell does
1. `cat = load_catalog()` reads the JSON file and returns the list under `"models"`: eleven dictionaries, one per row.
2. The first `print` writes the column headings. It uses an **f-string** with **format specs**: `{'model':46}` pads the text to 46 characters; `{'ctx':>8}` pads to 8 and aligns right (`>`). That is how the columns line up.
3. The `for` loop visits each row `m`. Two **conditional expressions** (`A if condition else B`, Python's one-line if/else) build the price text: `"?"` when the price is `None` (unknown), otherwise the number with 3 decimals (`:.3f`) for input or 2 (`:.2f`) for output.
4. The long `print` writes one line per row. `m['name'][:46]` cuts long names to 46 characters (that is why some names end mid-word, like `...GPU serve`). `str(m['open_weight'])[:1]` turns `True`/`False` into `T`/`F`. `:8d` prints a whole number in 8 places; `:4.1f` a decimal with one digit.
5. Nothing is sent anywhere: this is a file on your disk.

In [ ]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

### What you should see
This output comes from the file in the repo, not from a model, so yours should match exactly (unless the instructor has updated the catalog since):
```
model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)         vendor-api  F       200000  2.000  10.00  4  3.5
Frontier economy tier (closed, vendor API)     vendor-api  F      1000000  0.500   2.50  3  1.5
```
**How to read it, column by column:**
- `open`: every open-weight row is `T`; the three frontier rows are `F`. Only `T` rows can ever be self-hosted.
- `in$` / `out$`: output costs more than input in every priced row: between about 2 and 5 times more in this catalog. The self-hosted and laptop rows show `0.000`: no per-token bill. Their cost is elsewhere (Question 1).
- `?` in the Llama 3.1 8B row: the output price is blank **on purpose** (exercise 1).
- `Q` (quality 1–5) and `s` (latency in seconds) are **guesses**. Two of them become your measurements in section 4.
- `ctx`: most rows can see 131,072 tokens. Bigger is not automatically better: you pay for every token you put in.

The JSON file has more columns than this table shows: `reasoning`, `vision`, `fixed_monthly_usd`, `notes`, and for the laptop row `max_calls_per_day`. Open `module01/models_catalog.json` in VS Code and read the `notes` of three rows: they say what is an assumption.

<details><summary><b>▸ Read more: where do these numbers come from, and how stale can they get?</b></summary>

Prices come from the providers' pricing pages and from third-party trackers that collect them. Trackers are convenient, but they lag behind price changes and sometimes disagree with each other. The three frontier rows are not single models; they are *tiers* summarising a range (the flagship notes say roughly 2–10 USD in and 10–50 USD out per million tokens in September 2026 reports). The self-hosted rows contain an explicit **ASSUMPTION** about the monthly cost of a GPU server: replace it with a real quote before using it in a business case. A professional habit: every number in a decision document needs a source and a date. If you cannot trace it, mark it as a guess.
</details>

### Exercise 1 and exercise 2

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).
**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.

### What the next cell does
1. It creates `OVERRIDES`, an empty **dictionary**. Each key will be a row's `id`; each value, a small dictionary of the fields you want to replace.
2. The line starting with `#` is a **comment**: Python ignores it. It shows the shape to use.
3. To do exercise 1: remove the `#` and the space after it, and replace `???` with the price you found, as a plain number (no `$`, no quotes). For example, a price of 0.25 USD per million is written `0.25`. Keep the comma at the end of the line.

The cell prints nothing. Sections 3 and 5 read `OVERRIDES`.

In [ ]:
OVERRIDES = {
    # "groq-llama31-8b": {"price_out": ???},      # <- exercise 1: replace ??? with the number you found
}

### What you should see
Nothing. If you uncommented the line but left `???`, you get `SyntaxError`: Python cannot read `???` as a number. Replace it, or put the `#` back.

| If you see... | It means... | Do this |
|---|---|---|
| `SyntaxError: invalid syntax` | `???` is still there, or a `$` or a quote is in the number | type only the number, e.g. `0.25` |
| Section 3 still prints `price unknown` for the 8B row | the line is still a comment, or the `id` is misspelt | the key must be exactly `"groq-llama31-8b"` |
| Section 5 still excludes the 8B row in `faq` | after filling the price, other constraints still apply | read the reasons printed next to `x` |

### Check yourself
*Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?*

(Hint: look at the field `fixed_monthly_usd` in the JSON file, and at the word ASSUMPTION in the notes.)

> **Common mistake:** editing `models_catalog.json` itself. The catalog is shared course material; `git pull` may overwrite your change, or refuse to run because of it. Put your numbers in `OVERRIDES`: they are visible, reversible, and saved in your notebook.

## 3. Three jobs, three bills

### What and why
One product, three scenarios with different volumes and different rules. `SCENARIOS` holds them; edit any number to test "what if".

- **faq** — public product questions. Cheap and high-volume; no customer data.
- **documents** — reads a customer's salary slips and statements. **Customer PII: data must stay in-house.**
- **agent_assist** — live suggestions to a call-centre agent. **Must answer within about 3 seconds.**

Before you compare models on quality, you need to know what each one would cost *for this job*. A price card says "0.075 USD per million input tokens"; a manager asks "how much per month?" This section turns the first into the second, for every row and every job.

### The idea: price the task, not the token
```
 calls per month   = calls_per_day × 30                      (DAYS = 30)
 input tokens      = calls per month × in_tokens per call
 output tokens     = calls per month × out_tokens per call
 cost in USD       = (input tokens × price_in + output tokens × price_out) / 1,000,000  + fixed_monthly_usd
 cost in rupees    = cost in USD × 84                         (USD_TO_INR = 84, an assumption: edit it)
```
The same formula from the deck, worked for the `faq` job on gpt-oss-20b:
```
 20,000 calls/day × 30 days       = 600,000 calls
 input   600,000 × 1,200 tokens   = 720M tokens  × 0.075 USD per 1M  =    54 USD
 output  600,000 ×   200 tokens   = 120M tokens  × 0.30  USD per 1M  =    36 USD
                                                                total =    90 USD  ≈  Rs 7,560 per month
 same job on the flagship tier:   720 × 5.00  +  120 × 25.00          = 6,600 USD ≈  Rs 5,54,400 per month
```
About **73 times** apart, for the same job. That is why tiers exist.

**Analogy — the electricity bill.** The price per unit is printed on the meter card, but what you pay depends on how many units you use. And some homes also pay a fixed monthly charge, whatever they use. Self-hosted rows are all fixed charge and no per-unit price.

**The scenarios, side by side** (from `landscape_utils.py`):

| Job | Calls/day | Tokens in / out per call | Hard limits | Weights (cost / latency / quality) |
|---|---|---|---|---|
| `faq` | 20,000 | 1,200 / 200 | context ≥ 8,000; quality ≥ 2 | 0.6 / 0.2 / 0.2 |
| `documents` | 2,000 | 6,000 / 500 | **on-prem only**; context ≥ 16,000; quality ≥ 3 | 0.2 / 0.2 / 0.6 |
| `agent_assist` | 30,000 | 1,500 / 150 | **≤ 3.0 s**; context ≥ 8,000; quality ≥ 3 | 0.2 / 0.5 / 0.3 |

**Predict first:** which job has the biggest bill on gpt-oss-20b? Which has the smallest? (Multiply calls by tokens in your head before you run.)

### What the next cell does
1. `SCENARIOS.items()` gives each key and value as a pair, so `for key, sc in ...` gets the name (`"faq"`) in `key` and the scenario dictionary in `sc`.
2. It prints one heading per job. `{sc['calls_per_day']:,}` adds thousands separators (`20,000`). Python uses Western grouping, so it prints `554,400`, which is the same number the deck writes as `5,54,400`.
3. Inside, for each catalog row `m`: `{**m, **OVERRIDES.get(m["id"], {})}` builds a **new dictionary**: first all of the row's fields, then your override fields on top (later keys win). `OVERRIDES.get(key, {})` is **`dict.get` with a default**: it returns your override if there is one, or an empty dictionary if not, instead of raising `KeyError`.
4. `monthly_cost_inr(m, sc)` applies the formula above and returns whole rupees, or `None` if a price is unknown.
5. A conditional expression prints `price unknown` for `None`, otherwise the amount right-aligned in 12 places with separators (`:>12,`).

In [ ]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))

### What you should see
This is arithmetic on the catalog, so these are the real values for the starting catalog (with no overrides). Two of the three blocks, shortened:
```
faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        7,560/month
   ...
   Llama 3.1 8B Instant on Groq                   price unknown
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       58,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      168,000/month
   Frontier flagship (closed, vendor API)         Rs      554,400/month
   ...
documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq                            Rs        3,024/month
   ...
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       58,800/month
```
**How to read it:**
- The **self-hosted rows cost the same in every job** (Rs 58,800 and Rs 1,68,000): the fixed server cost does not depend on volume. The per-token rows grow with volume.
- Compare the two gpt-oss-20b rows in `documents`: Rs 3,024 rented versus Rs 58,800 self-hosted, roughly **19 times more**. Keep that number: it becomes the *price of residency* in section 5.
- The laptop shows **Rs 0**. It is not in any bill, but it will still lose the `faq` job in section 5. (Why? Read its `notes` in the JSON file.)
- If you filled exercise 1, the 8B row now shows a number instead of `price unknown`.

**Try a "what if"** (optional): in a new cell, run `SCENARIOS["faq"]["calls_per_day"] = 2_000`, then re-run this cell. Which bills drop by ten times, and which do not move at all? (`2_000` is the same as `2000`; Python allows `_` in numbers for readability.) To go back to the original numbers, **restart the kernel** and run from the top. Re-running the setup cell is not enough: your line changed the dictionary that lives inside `landscape_utils` itself, and `import` hands you that same, changed dictionary again.

### Under the hood: rebuild the monthly cost
`monthly_cost_usd` is about six lines. You write it yourself and check that it gives exactly the same answer as the real helper for **every row in every job** (33 pairs). No key, no network.

**What the next cell does:**
1. `my_monthly_cost_usd(model, sc)` returns `None` if either price is `None` (unknown). Otherwise it computes calls per month, the variable part (tokens × price / 1,000,000), and adds `fixed_monthly_usd`. `model.get("fixed_monthly_usd") or 0` uses **`dict.get`** (no error if the field is missing) and `or 0` (use 0 if the value is missing or `None`).
2. A **list comprehension** (`[... for ... in ... for ... in ...]`, a compact loop that builds a list) checks every scenario × row pair and keeps `True`/`False` for "same as the real helper".
3. `sum(checks)` counts the `True` values (Python treats `True` as 1).
4. It prints the deck's hand calculation for `faq`, so you can see the 720M and 120M tokens.

In [ ]:
from landscape_utils import monthly_cost_usd, DAYS, USD_TO_INR

def my_monthly_cost_usd(model, sc):
    if model.get("price_in") is None or model.get("price_out") is None:
        return None                                                    # unknown price: say so, never guess
    calls = sc["calls_per_day"] * DAYS                                 # 30 days a month
    variable = (calls * sc["in_tokens"] * model["price_in"] + calls * sc["out_tokens"] * model["price_out"]) / 1_000_000
    return round(variable + (model.get("fixed_monthly_usd") or 0), 2)  # + the server bill, if any

checks = [my_monthly_cost_usd(m, sc) == monthly_cost_usd(m, sc["calls_per_day"], sc["in_tokens"], sc["out_tokens"])
          for sc in SCENARIOS.values() for m in cat]
print(f"{sum(checks)} of {len(checks)} (row, job) pairs give the same cost ->", "same" if all(checks) else "DIFFERENT")

faq = SCENARIOS["faq"]
calls = faq["calls_per_day"] * DAYS
print(f"faq: {calls:,} calls/month · {calls * faq['in_tokens'] / 1e6:,.0f}M tokens in · {calls * faq['out_tokens'] / 1e6:,.0f}M tokens out")
for m in cat:
    if m["id"] in ("groq-gpt-oss-20b", "selfhost-gpt-oss-20b", "frontier-flagship"):
        usd = my_monthly_cost_usd(m, faq)
        print(f"   {m['name'][:46]:46} {usd:>9,.2f} USD  = Rs {round(usd * USD_TO_INR):>9,}")

**What you should see:** `33 of 33 (row, job) pairs give the same cost -> same`, then `faq: 600,000 calls/month · 720M tokens in · 120M tokens out`, and three lines: 90.00 USD (Rs 7,560), 700.00 USD (Rs 58,800), 6,600.00 USD (Rs 554,400). Change `DAYS` to `31` inside `my_monthly_cost_usd` and re-run: `DIFFERENT`. That is how you know the check is real.

**The take-away:** a monthly bill is three multiplications and a fixed charge. If a number in a business case looks surprising, you can now recompute it in one minute.

<details><summary><b>▸ Read more: four traps the formula does not show</b></summary>

1. **Output is dearer than input**, so "answer in two lines" in the system prompt is a cost instruction, not just politeness.
2. **Reasoning tokens are billed as output**, although you never see them. A reasoning model's real `out_tokens` per call can be much larger than the visible answer. Today's scenarios use the visible size, so treat reasoning rows as *underestimated*.
3. **Discounts exist:** providers can cache a repeated prompt beginning (a long, fixed system prompt) and charge less for it, and batch jobs that can wait are often about half price. Ask before you pay.
4. **The cost of a finished task includes retries.** A cheap model that needs three attempts can cost more than a dearer one that is right the first time. You can only know by measuring quality, which is section 4 and Session 17.
</details>

### Check yourself
*Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?*

> **Common mistake:** comparing price cards instead of bills. "0.075 versus 5.00" says nothing about whether the difference is Rs 500 or Rs 5 lakh a month; only volume × tokens × price tells you. Always multiply before you argue.

## 4. Measure — the same prompt on a hosted model and on your laptop

### What and why
Catalog numbers are a prior; **your measurements are evidence.** We send one prompt to Groq and to Ollama, time it, and check the arithmetic.

- **TTFT** — time to first token you can see. For a reasoning model this includes the hidden thinking.
- **tokens/s** — output speed.
- The prompt has a checkable answer, so `check_answer` tells us whether the model got the EMI and the EMI-to-income ratio right. That is a tiny **evaluation**; S17 builds the full version.

Why a loan question? Because it is the kind of thing the `documents` and `agent_assist` jobs will really be asked, and because the right answer can be computed in Python. A question with a checkable answer turns "this answer looks good" into "this answer is right or wrong".

### The idea: three numbers for speed, one for correctness
```
 send ──┬───────── TTFT ─────────┬──────────── generating the visible answer ────────────┬─► done
        │  (network + queue +    │ first visible token                                   │
        │   hidden thinking)     │                                                       │
        └────────────────────────┴───────────────────── total_s ─────────────────────────┘
 tokens_per_s = completion_tokens / total_s          (an average over the WHOLE call, thinking included)
```
**Analogy — the restaurant again.** TTFT is how long until the first dish reaches your table; tokens per second is how fast the other courses arrive; total time is how long until the meal is over. A reasoning model is a chef who thinks for a while before serving anything. A person chatting feels the first dish; a call-centre agent waiting for a complete suggestion feels the total.

**The question, and its right answer** (computed in Python by `emi()`):
- Loan Rs 8,00,000 at 11.5% a year for 60 months. Monthly rate `r = 11.5 / 12 / 100`.
- EMI = principal × r × (1 + r)^60 / ((1 + r)^60 − 1) = **Rs 17,594.09**.
- (EMI + existing Rs 8,000) / Rs 60,000 = **42.7%**: under 50%, so *yes*.

**Predict first:** which will reach the first visible token sooner, Groq (a 20B reasoning model on fast hosted hardware) or your laptop (a 3B model, no thinking)? Which will produce more tokens per second? Will both get the EMI right?

### What the next cell does
1. Prints the prompt, so you know exactly what both models receive.
2. `RESULTS = {}` starts an empty dictionary; each successful run is stored under its label.
3. The `for` loop walks a list of **tuples** `(label, client, model)` and unpacks each into three variables. The Groq model name comes from `.env`; the Ollama one is fixed: `"llama3.2:3b"`.
4. `if client is None:` prints `skipped (no client ...)`; `continue` jumps to the next item in the loop. With no key, Groq stops here.
5. `try:` calls **`time_stream(client, model)`**. It sends the prompt with `stream=True`, so the reply arrives in small pieces (*chunks*), and it records when the first visible piece arrived and when the last one did. It allows up to 1,500 output tokens, because a reasoning model spends tokens thinking first.
6. `r["check"] = check_answer(r["text"])` adds the arithmetic check to the result dictionary.
7. Prints the timings, the check, and the first 500 characters of the answer (`.strip()` removes blank lines at the ends; `[:500]` cuts the rest).
8. If anything fails (Ollama not running, a wrong key, a rate limit, no internet), `except Exception as e` prints `skipped (` + the error's type name (`type(e).__name__`) + the first 120 characters of the message, and the loop carries on. That is why this cell needed **no change** for the no-key rule.

In [ ]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

### What you should see
First the prompt. Then one block per model. *An illustration, not a real run* (every number below is made up; yours will differ, and that is the point):
```
--- groq: openai/gpt-oss-20b
first token 0.9 s | total 1.6 s | 420 tokens | 262.5 tokens/s
arithmetic check: {'emi_ok': True, 'ratio_ok': True, 'score_out_of_2': 2}
EMI ≈ Rs 17,594 per month.
Total EMIs = Rs 25,594, which is about 42.7% of income.
Yes, it is under 50%.

--- ollama: llama3.2:3b
first token 2.4 s | total 11.8 s | 120 tokens | 10.2 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
To calculate the EMI, we use the formula ...
```
With no key and no Ollama, this is the real output:
```
--- groq: skipped (no client - add your key to .env)

--- ollama: skipped (APIConnectionError: Connection error.)
```
**How to read it:**
- `first token` is TTFT; `total` is the whole call. On Groq, a large part of TTFT can be hidden thinking.
- `tokens` comes from the provider's usage report when it sends one. For a reasoning model it can include the hidden thinking tokens, so it may be much larger than the visible answer. If the provider sends no usage, `time_stream` counts text chunks instead, which is only an approximation.
- `tokens/s` divides by the total time, including the wait for the first token, so it is an average over the whole call, not pure writing speed.
- `arithmetic check` is a **hint, not a grade**: it only looks for numbers close to 17,594.09 and 42.7 anywhere in the text (see the under-the-hood cell below). You give the quality score yourself, with the rubric.

| If you see... | It means... | Do this |
|---|---|---|
| `groq: skipped (no client ...)` | no key in `.env` | add the key, restart, Restart & Run All |
| `groq: skipped (AuthenticationError ...)` | the key is wrong, revoked, or still the placeholder | paste the real key; restart |
| `groq: skipped (RateLimitError ...)` | `429`: the whole class hit the free tier at once | wait 30 seconds; re-run this cell only |
| `ollama: skipped (APIConnectionError ...)` | Ollama is not running or not installed | optional: start the Ollama app or run `ollama serve` in a second terminal |
| `ollama: skipped (NotFoundError ...)` | the model is not downloaded | `ollama pull llama3.2:3b` |
| Groq's answer is empty, check `0` | the reasoning model used the token budget thinking | rare at 1,500 tokens; tell the instructor |
| `emi_ok: False` but the answer says "about 17.6k" | the checker only reads plain numbers | grade it yourself with the rubric |

The first local run is often slower: Ollama loads the model into memory on the first request. If you have time, run the cell twice and use the second timing.

### Under the hood: how `check_answer` reads an answer
`check_answer` does not understand the answer. It pulls out every number, then asks: is any number within 1% of the right EMI, and is any number within 0.6 of the right ratio? You rebuild it and compare it with the real one on six practice answers. No model is called.

**What the next cell does:**
1. `my_emi(p, rate, months)` is the standard EMI formula; it must give 17594.09 (rounded), the value of `EXPECTED_EMI`.
2. `my_numbers(text)` uses a **regular expression** (a pattern for text, from the `re` module). `\d[\d,]*\.?\d*` means: a digit, then any digits or commas, then an optional dot and more digits. So `17,594.09` and Indian-style `8,00,000` are both read as one number. `.replace(",", "")` removes the commas before `float()` turns the text into a number.
3. `my_check(text)` uses **`any(...)`** with a generator: `True` if at least one number is close enough. `abs(n - EXPECTED_EMI)` is the distance from the right answer.
4. For six practice answers it prints the real result and yours, then `same` or `DIFFERENT`.

In [ ]:
import re
from landscape_utils import check_answer, emi

def my_emi(p, annual_rate_pct, months):
    r = annual_rate_pct / 12 / 100                                     # monthly rate as a fraction
    return p * r * (1 + r) ** months / ((1 + r) ** months - 1)

def my_numbers(text):
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", text) if x.replace(",", "").replace(".", "").isdigit()]

def my_check(text):
    nums = my_numbers(text)
    emi_ok = any(abs(n - EXPECTED_EMI) <= 0.01 * EXPECTED_EMI for n in nums)    # within 1% of 17,594.09
    ratio_ok = any(abs(n - EXPECTED_RATIO) <= 0.6 for n in nums)               # within 0.6 points of 42.7
    return {"emi_ok": emi_ok, "ratio_ok": ratio_ok, "score_out_of_2": int(emi_ok) + int(ratio_ok)}

print("my EMI:", round(my_emi(800_000, 11.5, 60), 2), "| real EMI:", round(emi(800_000, 11.5, 60), 2), "| EXPECTED_EMI:", EXPECTED_EMI)
practice = ["EMI is Rs 17,594.09. Total 25,594.09 is 42.7% of income. Yes, under 50%.",
            "EMI Rs 17,594; ratio 42.66%",
            "EMI about 17.6k; ratio 43%",
            "EMI ~ Rs 17594 per month",
            "The EMI is Rs 9,800 and the ratio is 29%.",
            ""]                                                        # practice answers, not real model output
for text in practice:
    real, mine = check_answer(text), my_check(text)
    print(f"{text[:44]!r:48} real={real['score_out_of_2']}/2 mine={mine['score_out_of_2']}/2 ->", "same" if real == mine else "DIFFERENT")

**What you should see:** the three EMI values agree (17594.09), then six lines ending in `same`, with scores 2, 2, 1, 1, 0, 0. Look at two of them:
- `"EMI about 17.6k; ratio 43%"` gets **1 of 2**: the EMI is right in meaning, but `17.6` is not within 1% of 17,594.09. And `43` counts as the ratio, because it is within 0.6 of 42.7.
- `"EMI ~ Rs 17594 per month"` gets 1 of 2: no ratio at all.

**The take-away:** an automatic checker is fast and consistent, but it checks *form* as much as *meaning*. Any number close to 42.7 anywhere in the text passes, even in a wrong sentence. That is why you also score with a human rubric, and why Session 17 builds a proper evaluation.

### Score the answers yourself
**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows).

**What the next cell does:** prints `RUBRIC`, a text stored in `landscape_utils.py`. Nothing else.

In [ ]:
print(RUBRIC)

### What you should see
The rubric, exactly as stored in the helper file:
```
Quality rubric (1-5), fill in yourself after reading the answer:
 5  EMI and ratio both right, exactly three short lines, clear yes/no on the 50% test
 4  Both numbers right but format or wording off
 3  One number right, or right numbers with a wrong conclusion
 2  Both numbers wrong but the method is visible
 1  Wrong and confident, or no answer
```
Read each model's answer from the cell above and give it a score. Use the `arithmetic check` as a hint, not as the score: the rubric also asks for the format (three short lines) and the conclusion (a clear yes or no).

### What the next cell does
1. `MEASURED` is a dictionary like `OVERRIDES`: key = the row's `id`, value = the fields to replace. The two lines are **comments** showing the shape; the numbers in them are examples, not results.
2. To fill it: remove the `#` from a line, put **your** `total_s` from section 4 as `latency_s` (total time is a good stand-in for latency), and your rubric score as `quality` (a whole number, 1 to 5).
3. `OVERRIDES.update(MEASURED)` copies your entries into `OVERRIDES`, so sections 5 and 6 use them.
4. The last line is just `OVERRIDES`. In Jupyter, the value of the **last expression** in a cell is displayed, so you see the final dictionary without `print`.

In [ ]:
# Fill in from YOUR run. total_s is a good stand-in for latency_s. quality: 1-5 from the rubric.
MEASURED = {
    # "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    # "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)
OVERRIDES

### What you should see
With nothing filled in, `{}`. After exercise 1 and two measured rows, something like this (*an illustration*: your price and your numbers):
```
{'groq-llama31-8b': {'price_out': ...}, 'groq-gpt-oss-20b': {'latency_s': 1.6, 'quality': 5}, 'ollama-llama32-3b': {'latency_s': 11.8, 'quality': 2}}
```
If Ollama was skipped, fill only the Groq line. If both were skipped, leave `MEASURED` empty: section 5 then uses the catalog's guesses, and you say so in section 6.

### Check yourself
*Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?*

> **Common mistake:** putting a price override and a measurement for the **same** row in two places. `OVERRIDES.update(MEASURED)` replaces the whole inner dictionary for that `id`, so a price you put in `OVERRIDES` for `groq-gpt-oss-20b` would be lost. Keep all fields for one row in one dictionary. A second trap: re-running the `OVERRIDES = {...}` cell (section 2) empties it again; after any change, use Restart & Run All.

## 5. The selection matrix

### What and why
For each scenario: **hard constraints first** (a model that fails one is out, however cheap), then a weighted score of cost, speed and quality. Weights are in `SCENARIOS`; change them and watch the winner move.

This is where everything from sections 2–4 meets: the catalog, your price from exercise 1, the bills, and your measurements. The output is one ranked list per job, plus a list of excluded models **with the reason for each**. In a review, the reasons matter as much as the winner.

### The idea: eliminate, then score, then verify
```
 11 catalog rows
      │
      ▼  1. ELIMINATE — hard constraints (pass or out; a weight cannot rescue a model)
      │     needs_on_prem? latency ≤ limit? calls/day ≤ capacity? context ≥ min? quality ≥ min? price known?
      ├──► x  excluded, with every reason
      ▼
 survivors
      │
      ▼  2. SCORE — among the survivors only, each of cost, latency, quality becomes 0..1 (1 = best)
      │     score = 100 × (w_cost × cost + w_latency × latency + w_quality × quality)
      ▼
 ranked list  ──►  3. VERIFY the top two on your own examples (today: one EMI prompt; S17: a golden set)
```
**Analogy — hiring.** Step 1 is eligibility: if the job needs someone based in the city, candidates elsewhere are out, however good their interview would have been. Step 2 is the interview scorecard, with weights the hiring manager sets. Step 3 is the trial task before the offer letter.

**Two details of the scoring that change results:**
- **Normalisation is among the survivors.** The cheapest survivor gets cost 1.0 and the dearest gets 0.0, whatever their real prices. So if only two models survive, one gets full marks and the other zero on cost and on latency, even if they are close.
- **Cost is compared on a log scale.** Costs here run from Rs 0 to over Rs 5 lakh a month. On a straight scale, one very expensive model would make all the cheap ones look the same. On a log scale, "ten times dearer" is always the same step.

### What the next cell does
1. `for key, sc in SCENARIOS.items():` visits each job.
2. `ranked, excluded = rank(sc, overrides=OVERRIDES)` — **`rank` returns two things at once** (a tuple), and Python **unpacks** them into two variables. `overrides=OVERRIDES` is a **keyword argument**: your price and measurements go in here.
3. `if not ranked:` — an empty list counts as false, so this prints a warning when nobody survives.
4. `for i, m in enumerate(ranked, 1):` — **`enumerate`** gives a counter with each item; the `1` makes the counter start at 1 instead of 0, so the list is numbered 1, 2, 3...
5. Each ranked line shows the score (`:5.1f`: one decimal, 5 places), the monthly cost in rupees, the latency and the quality.
6. For each excluded model, `'; '.join(why)` glues the list of reasons into one line with `; ` between them.

In [ ]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")

### What you should see
With the **starting catalog and no overrides**, this is the real output for the first job (it is arithmetic, so it is the same on every laptop; your overrides will move it):
```
=== faq: Public product FAQ chatbot   weights {'cost': 0.6, 'latency': 0.2, 'quality': 0.2}
  1. gpt-oss-20b on Groq                            score  90.0   Rs      7,560/month   1.5 s   Q3
  2. Llama 4 Scout 17B-16E on Groq                  score  86.0   Rs     10,080/month   1.5 s   Q3
  3. gpt-oss-120b on Groq                           score  83.1   Rs     15,120/month   2.0 s   Q4
  ...
  9. Frontier flagship (closed, vendor API)         score  20.0   Rs    554,400/month   6.0 s   Q5
  x  Llama 3.1 8B Instant on Groq                   a price is unknown - fill it in
  x  llama3.2:3b on your laptop (Ollama)            cannot serve 20,000 calls/day (this setup tops out near 5,000)
```
**How to read it:**
- Read the `x` lines first. They are the hard constraints at work. The laptop is free per token and still out: **capacity is a hard constraint too**. The 8B row is out only because its price is unknown; fill exercise 1 and it enters the race.
- In `documents`, almost every row is excluded with `data must stay on-prem; this model's provider sees the prompt`. Only the rows with `runs_on_prem = true` can enter, and the laptop also fails on quality.
- In `agent_assist`, look for `too slow (... s > 3.0 s)`. Which rows fail only on speed?
- Compare the winners across the three jobs. **The same weights can win two jobs in two different places**: rented for one job, self-hosted for another. Where a model runs is a separate decision from which model.

| If you see... | It means... | Do this |
|---|---|---|
| `nobody survives the hard constraints` | your overrides or edits made every row fail | read the `x` reasons; relax one constraint or fix an override |
| a row you measured is now excluded for latency | your measured `latency_s` is above the job's limit | that is evidence working; keep it |
| scores changed after you filled `MEASURED` | normalisation is among survivors: one row's new number moves everyone's score | expected; compare the order, not the exact score |

### Under the hood: rebuild the score
You rebuild the scoring part of `rank` and check it gives the same scores as the real helper, **with your overrides**, for all three jobs. For the pass/out step it reuses the real `eligible()`, whose six rules you saw in the diagram above.

**What the next cell does:**
1. `norm(v, lo, hi)` turns a value into 0..1 where the **lowest** value gets 1 (best). If all values are equal, everyone gets 1.
2. `my_rank(sc, overrides)` merges each row with its override, keeps the rows that pass `eligible()`, computes their monthly cost, then normalises cost on a log scale (`math.log1p(x)` is log(1 + x), which also works for a cost of 0), latency on a straight scale, and quality as `(Q − 1) / 4` (1 → 0.0, 5 → 1.0). It multiplies by the weights and by 100.
3. `sorted(..., key=lambda m: m["score"], reverse=True)` sorts by score, highest first. A **`lambda`** is a one-line function without `def`: here, "given a row, give back its score".
4. It compares your list of `(id, score)` pairs with the real one for each job.
5. For `documents` it prints each survivor's three parts, so you can see what normalisation among only two survivors does.

In [ ]:
import math
from landscape_utils import eligible

def norm(v, lo, hi):                                                   # lowest value -> 1.0 (best), highest -> 0.0
    return 1.0 if hi == lo else 1 - (v - lo) / (hi - lo)

def my_rank(sc, overrides=None):
    ok = []
    for base in load_catalog():
        m = {**base, **(overrides or {}).get(base["id"], {})}           # your override wins
        if eligible(base, sc, overrides)[0]:                           # step 1: pass or out
            m["cost_usd"] = monthly_cost_usd(m, sc["calls_per_day"], sc["in_tokens"], sc["out_tokens"])
            ok.append(m)
    if not ok:
        return []
    costs = [math.log1p(m["cost_usd"]) for m in ok]; lats = [m["latency_s"] for m in ok]
    w = sc["weights"]
    for m in ok:                                                       # step 2: score among the survivors only
        m["parts"] = (norm(math.log1p(m["cost_usd"]), min(costs), max(costs)),     # cost, on a log scale
                      norm(m["latency_s"], min(lats), max(lats)),                  # latency, straight scale
                      (m["quality"] - 1) / 4)                                      # quality 1..5 -> 0..1
        m["score"] = round(100 * (w["cost"] * m["parts"][0] + w["latency"] * m["parts"][1] + w["quality"] * m["parts"][2]), 1)
    return sorted(ok, key=lambda m: m["score"], reverse=True)

for key, sc in SCENARIOS.items():
    mine = [(m["id"], m["score"]) for m in my_rank(sc, OVERRIDES)]
    real = [(m["id"], m["score"]) for m in rank(sc, overrides=OVERRIDES)[0]]
    print(f"{key:13} {len(mine)} survivors   ->", "same" if mine == real else "DIFFERENT")

print("\ndocuments, part by part   (cost, latency, quality: 1.0 = best among the survivors)")
for m in my_rank(SCENARIOS["documents"], OVERRIDES):
    c, l, q = m["parts"]
    print(f"   {m['name'][:46]:46} cost {c:.2f}  latency {l:.2f}  quality {q:.2f}  -> score {m['score']}")

**What you should see:** three lines ending in `same`. Then, with the starting catalog, two survivors in `documents`:
```
   gpt-oss-20b self-hosted on a smaller GPU serve cost 1.00  latency 1.00  quality 0.50  -> score 70.0
   gpt-oss-120b self-hosted on your own GPU serve cost 0.00  latency 0.00  quality 0.75  -> score 45.0
```
(If your overrides changed these rows, your numbers differ.) Quality weighs 0.6 in this job, and the 120b row has the better quality guess (Q4 against Q3). It still loses: with only two survivors, normalisation gives the cheaper, faster one full marks and the other zero on cost and latency. Work it out: 100 × (0.2 × 1 + 0.2 × 1 + 0.6 × 0.5) = 70; 100 × (0 + 0 + 0.6 × 0.75) = 45.

**The take-away:** a scoring rule is a set of choices (which scale, normalised among whom, which weights). The ranking is only as good as those choices, so you read the parts, not just the total, before you defend a winner.

<details><summary><b>▸ Read more: why normalising among the survivors can surprise you</b></summary>

Because every score is relative to the best and worst survivor, **adding or removing one model can change the order of two others**. For example, if a new, very expensive model passes the constraints, it becomes the "worst cost" anchor, and the cost gaps between the cheaper models shrink in comparison. Decision analysts call this kind of effect *rank reversal*. It does not make the method wrong; it means that (1) you should always print the parts, as above; (2) you should re-run the matrix whenever the catalog changes; and (3) for a close race, the verify step (your own examples) decides, not the third decimal of a score. Alternatives exist, such as fixed scales (for example, "Rs 0 = 1.0, Rs 5 lakh = 0.0" agreed in advance), which keep scores stable when the list changes.
</details>

### Check yourself
*Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?*
*Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?*

To do Question 5, add a new cell below the matrix cell and run `SCENARIOS["faq"]["weights"] = {"cost": 0.1, "latency": 0.2, "quality": 0.7}` (the weights should add up to 1), then re-run the matrix cell. **Predict first** which row will win. Afterwards, restart the kernel and run from the top before section 6, so your saved file uses the original weights, or say in section 6 which weights you used.

> **Common mistake:** treating the top score as "the answer". The score depends on guesses (quality, latency) and on weights someone chose. The matrix gives you a **shortlist and the reasons**; the decision still needs the verify step and a person who owns the weights.

## 6. Your decision — this is what you submit

### What and why
For each scenario write: the model you would choose, the runner-up, and **what evidence would change your mind**. Also write what you found when you verified two catalog rows (exercise 2), and answers to questions 1–5.

A choice without its reasons is not useful to anyone. Next month a price changes or a new model appears; whoever reads your decision must be able to see *which* assumption changed and whether the decision still holds. "What evidence would change your mind" is exactly that: the condition under which you would choose again.

### The idea: a decision record
For each job, a good answer has four parts. Here is the **shape**, with placeholders (not a model answer):
```
 documents:  CHOICE      <model> — because <the constraint it passes> and <the score parts that won>
             RUNNER-UP   <model> — loses on <cost / speed / quality>
             ASSUMPTIONS <which numbers are guesses: e.g. the GPU server cost, my one quality score>
             WOULD CHANGE IF  <a measurable condition, e.g. "a golden-set test shows quality below X">
```
**Analogy — the doctor's note.** Not just the prescription, but why this medicine, what else was considered, and when to come back.

### Write your answers here
Double-click this cell to edit it. Keep each answer short: two to four lines.

**faq:** 

**documents:** 

**agent_assist:** 

**Verified rows (exercise 2):** 

**Q1–Q5:** 

Then save and submit: run the cell below, and upload `module01/s10_matrix_output.md` through the submission form (personal repositories start at Session 14).

### What the next cell does
1. **`Path(landscape_utils.__file__)`** — `__file__` is the full path of the helper file on your disk; `Path` (from **`pathlib`**) turns it into a path object. **`.with_name("s10_matrix_output.md")`** keeps the folder and changes the file name. So the output always lands in `module01/`, whichever folder Jupyter runs in.
2. `lines` starts as a list of two text lines; `lines.append(...)` adds one line at a time.
3. For each job it calls `rank` again (with your overrides) and adds the ranked and excluded lists. `{m['cost_inr']:,}` adds thousands separators.
4. The "Measured" line uses a **nested dictionary comprehension**: for each label in `RESULTS` (`groq`, `ollama`), it keeps only four fields (`ttft_s`, `total_s`, `tokens_per_s`, `check`) and leaves out the long answer text.
5. `"\n".join(lines)` glues the lines with line breaks; `OUT.write_text(..., encoding="utf-8")` writes the file. Writing UTF-8 explicitly matters on Windows, where the default encoding can differ.
6. It prints where the file was saved.

In [ ]:
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   # next to the helper, whichever folder Jupyter runs in
lines = ["# Session 10 - selection matrix output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append("\n## Measured\n" + str({k: {kk: v[kk] for kk in ('ttft_s','total_s','tokens_per_s','check')} for k, v in RESULTS.items()}))
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)

### What you should see
One line with the full path, for example (*an illustration*; your folder will differ):
```
saved C:\Users\you\Documents\genai-agents-course\module01\s10_matrix_output.md
```
Open the file in VS Code and check it: three `##` sections, one per job, with ranked and excluded models, then a `## Measured` line. If section 4 was skipped, that line is `{}`: say so in your decision text. The file contains only what the code computed; your written answers stay in this notebook, so save the notebook too (Ctrl+S).

| If you see... | It means... | Do this |
|---|---|---|
| `NameError: name 'RESULTS' is not defined` | section 4 was not run in this kernel | Restart & Run All |
| `PermissionError` | the file is open in another program that locks it | close it there; re-run |
| the file shows your Question-5 weights | you changed `SCENARIOS` and did not restart | restart, Run All, re-run this cell, or explain it in your answer |

Then run the checkpoint in the terminal (PowerShell, course folder):
```
uv run pytest tests/test_s10.py
```
Expected: `8 passed`. These tests are all offline, so a key or Ollama makes no difference.

### Check yourself
*(New question.) Your `faq` choice wins mostly because of cost. Name one measurement that, if it came out badly, would make you switch to your runner-up, and say how you would collect it.*

> **Common mistake:** writing "gpt-oss-20b because it is the best". Best at what, under which constraint, with which weights? Name the constraint it passed, the score parts that won, and the assumption that would change your mind.

---
# Part C · Wrap-up

## C1. Summary
- There are **three ways to get a model**: a vendor API (closed; your text goes to the vendor), hosted open-weight (published weights run by a provider; your text goes to the provider), and self-hosted (the same open weights on your own machine; your text stays). Who sees the data, who bills you, and who is on call differ for each.
- A catalog row is part fact, part guess, part missing. Prices are **indicative** and go stale; quality and latency are starting guesses until you measure them. Fill gaps in `OVERRIDES`, never by guessing silently.
- **Price the task, not the token:** monthly cost = calls × tokens × price / 1,000,000 + fixed monthly cost. The same FAQ job was about 73 times apart between gpt-oss-20b and the flagship tier; self-hosting the same weights for the PII job cost about 19 times more than renting them: the price of residency.
- **Speed is three numbers:** time to first token, tokens per second, total time. A reasoning model's hidden thinking delays the first visible token and is billed as output. "Fast" belongs to a model *in a place*, not to the model alone.
- **Eliminate, then score, then verify.** Hard constraints (residency, latency, capacity, context, minimum quality, known price) remove options; a weighted score ranks the survivors; your own examples decide.
- The weights are a **business decision**, and normalisation among survivors means one change can move several scores. Read the parts, not just the total.
- A decision record names the choice, the runner-up, the assumptions, and **the evidence that would change your mind**.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Vendor API (closed model) | A model only its vendor runs; you rent it per token and cannot download it |
| Frontier, family, tier | The most capable models; one vendor's line-up; flagship / mid / economy sizes |
| Open-weight | The trained numbers are downloadable, under a licence (not the same as open-source) |
| Hosted open-weight vs self-hosted | Open weights run by a provider, or by you where the data lives |
| Parameters | The learned numbers in a model; 3B, 20B, 120B = billions |
| Mixture-of-experts | Many parameters in total, only a few active per token: fast for its size |
| Context window, max output tokens | What the model can see at once; the separate ceiling on one answer |
| Price per million tokens | Input and output priced separately; output usually costs more |
| Reasoning tokens | Hidden thinking, billed as output, delaying the first visible token |
| TTFT | Time until the first visible token |
| Tokens per second, total time | Output speed; the time for the whole answer |
| Benchmark, leaderboard, contamination | A public test and its ranking; seen-the-paper-before. Shortlist, don't decide |
| Hard constraint | Pass or out, before any scoring |
| Data residency | Where data may be processed or stored; set by compliance |
| Weighted score, normalisation | Cost, speed, quality on 0..1 among survivors, combined with business weights |
| Catalog value vs measurement | A prior from someone else versus evidence from your own run |

## C3. Interview practice

**1. "How would you choose an LLM for a new use case?"**
*Model answer:* In three steps. First eliminate with hard constraints: where the data may go, the latency budget, the volume, the context needed, a minimum quality. Then rank the survivors on cost, speed and quality with weights the business owner agrees to. Then verify the top two on a set of my own examples before committing, and write down what would make me choose again. *In this lab:* the `documents` job lost every hosted and vendor model to the on-prem constraint before any scoring; in `faq`, the laptop model was free per token but could not serve 20,000 calls a day. *Honest limit:* the ranking depends on guessed quality and latency values and on chosen weights; with only a handful of examples, the verify step is a sanity check, not proof.

**2. "What is the difference between open-weight, open-source and closed models? When would you self-host?"**
*Model answer:* Closed models are run only by their vendor. Open-weight models publish their trained weights, so anyone can run them; open-source would also release code and training data, and open-weight licences can restrict use. I would self-host when data cannot leave our environment, or when volume is so high and steady that a fixed server cost beats per-token pricing, and the team can operate and secure it. *In this lab:* gpt-oss-20b appeared twice, rented on Groq and self-hosted; the self-hosted row cost about Rs 58,800 a month against about Rs 3,024 rented for the documents job, roughly 19 times more, and was the only kind allowed. *Honest limit:* the self-hosted cost in the catalog is an assumption; a real business case also needs people, monitoring, security and upgrades.

**3. "How do you estimate the monthly cost of an LLM feature?"**
*Model answer:* Calls per month times input tokens times the input price, plus calls times output tokens times the output price, divided by a million, plus any fixed cost. Then check the traps: output usually costs more, reasoning models bill hidden thinking as output, caching and batch discounts may apply, and retries multiply the bill. *In this lab:* the FAQ bot at 600,000 calls a month came to 90 USD (about Rs 7,560) on gpt-oss-20b and 6,600 USD on the flagship tier. *Honest limit:* the token counts per call are estimates; a reasoning model's real output tokens can be much higher, so I would measure real usage in a pilot.

**4. "Can you trust a leaderboard to pick a model?"**
*Model answer:* To shortlist, yes; to decide, no. Models may have seen the test questions (contamination), tasks are narrow, some scores are reported by the vendor, and none of them use my documents, language or tools. I shortlist from public scores, then test on my own examples. *In this lab:* one prompt with a checkable answer (the EMI and the 42.7% ratio) plus a human rubric; the automatic checker gave "17.6k" only half marks, which shows why a human looks too. *Honest limit:* one prompt is a tiny sample; Session 17 builds a proper golden set.

**5. "Which latency numbers matter for a chat application, and how does a reasoning model change them?"**
*Model answer:* Time to first token (what a chat user feels), tokens per second (how fast the text flows), and total time (what a system waiting for the full answer feels). A reasoning model thinks before its first visible token, so its TTFT and total time rise, and its hidden tokens count as output. *In this lab:* `time_stream` measured TTFT, total time and tokens per second for Groq and for llama3.2:3b on the laptop; `agent_assist` excluded every row whose latency was over 3 seconds. *Honest limit:* one measurement from one laptop and one network is noisy; production numbers need many requests and percentiles (Session 53).

## C4. Troubleshooting

| Error or symptom | Cause | Fix |
|---|---|---|
| `no hosted client: Missing credentials...` | no `.env`, or no `API_KEY` in it | `copy .env.example .env` in PowerShell, paste your key (no quotes, no trailing space), restart the kernel |
| section 4: `groq: skipped (AuthenticationError ...)` | wrong or revoked key, or still `paste_your_groq_key_here` | paste a valid key; restart |
| `404 model_not_found`, and the model name in the error has extra text after it | a comment on the same line as `MODEL=` in `.env` | put comments on their own line; restart |
| section 4: `ollama: skipped (APIConnectionError ...)` | Ollama not running or not installed | optional: start the Ollama app or run `ollama serve`; the matrix still runs |
| section 4: `ollama: skipped (NotFoundError ...)` | model not downloaded | `ollama pull llama3.2:3b` |
| section 4: `RateLimitError` / `429` | free-tier burst when the whole class runs at once | wait 30 seconds; re-run that cell only |
| Groq answer is empty | the reasoning model used its token budget thinking | rare at 1,500 tokens; report it rather than editing the helper |
| `check_answer` says `False` for a correct answer | the number is written as `17.6k` or in words | grade by the rubric yourself; the checker is a hint |
| `SyntaxError` in the `OVERRIDES` cell | the `???` placeholder is still there | replace it with a number, or comment the line again |
| `price unknown` still printed after exercise 1 | the line is still a comment, or the id is misspelt | the key must be exactly `"groq-llama31-8b"` |
| `nobody survives the hard constraints` | an override or a scenario edit made every row fail | read the `x` reasons; relax one constraint |
| results still use changed weights after "re-running setup" | the edit changed the dictionary inside `landscape_utils` | restart the kernel and Run All |
| `ModuleNotFoundError: landscape_utils` | repo out of date, or notebook opened from another folder | `git pull`; open the notebook from the course folder |

## C5. What comes next
**Session 11 — Multi-provider API calls:** one `ask()` function for Groq, Gemini and Ollama, switching provider by editing `.env` only; the messages format and roles; temperature, `max_tokens` and stop sequences. Keep:
- your `.env` with the Groq lines active; the instructor will announce a second provider (create its free key before the session if you can);
- Ollama with `llama3.2:3b`, if you have it;
- today's habit: a model is a *choice for a job*, and the code around it should not care which one you chose. Session 11 builds exactly that code.

Before Session 11: notebook run top to bottom, `uv run pytest tests/test_s10.py` → `8 passed`, section 6 written, and `s10_matrix_output.md` uploaded through the form.